# 03 RNN 从零开始

目标：不用 `nn.RNN`，直接写出

```text
h_t = tanh(x_t W_xh + h_(t-1) W_hh + b)
```

观察 hidden state 如何沿序列传播。


In [ ]:
import torch

torch.manual_seed(42)

batch_size = 2
seq_len = 4
input_size = 3
hidden_size = 5

X = torch.randn(batch_size, seq_len, input_size)

W_xh = torch.randn(input_size, hidden_size, requires_grad=True)
W_hh = torch.randn(hidden_size, hidden_size, requires_grad=True)
b_h = torch.zeros(hidden_size, requires_grad=True)

H = torch.zeros(batch_size, hidden_size)

outputs = []

for t in range(seq_len):
    X_t = X[:, t, :]

    H = torch.tanh(
        X_t @ W_xh
        + H @ W_hh
        + b_h
    )

    outputs.append(H)

Y = torch.stack(outputs, dim=1)

print("X:", X.shape)
print("Y:", Y.shape)
print("final H:", H.shape)

## 每个时间步都共享同一组参数

In [ ]:
print("W_xh:", W_xh.shape)
print("W_hh:", W_hh.shape)
print("b_h:", b_h.shape)

## BPTT：损失可以从最后一步反传到更早时间步参数

In [ ]:
loss = Y[:, -1, :].pow(2).mean()
loss.backward()

print("loss:", float(loss))
print("W_xh grad norm:", float(W_xh.grad.norm()))
print("W_hh grad norm:", float(W_hh.grad.norm()))

## 与 `nn.RNN` 的关系

`nn.RNN` 帮你封装了时间循环、参数管理、层数等细节；上面的核心递推思想没有变化。


In [ ]:
from torch import nn

rnn = nn.RNN(
    input_size=input_size,
    hidden_size=hidden_size,
    batch_first=True,
)

output, h_n = rnn(X)

print(output.shape)
print(h_n.shape)